# Notebook 02 — Transform to Silver Layer

Clean, type-cast, rename, and deduplicate the raw bronze data into analysis-ready
Silver tables. Uses Delta Lake **MERGE** (upsert) for players and player history
so re-runs update existing rows rather than creating duplicates.

**Key transformations**: column renaming for clarity, type casting (String → Int/Double),
deduplication by primary key, and incremental upserts via `whenMatchedUpdateAll`.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType
from delta.tables import DeltaTable

## Silver Players

Select the most recent ingestion snapshot from bronze, rename API field names
to friendlier aliases (e.g. `web_name` → `display_name`, `now_cost` → `current_price`),
cast numeric strings to proper types, and deduplicate by `player_id`.

Uses a MERGE/upsert pattern: if the silver table already exists, matched rows
are updated and new players are inserted. On the very first run it creates the table.

In [ ]:
# Read bronze and keep only the latest ingestion snapshot
bronze_players = spark.table("fpl_project.bronze.players")
latest_ts = bronze_players.agg(F.max("_ingestion_ts")).collect()[0][0]

silver_players = bronze_players \
    .filter(F.col("_ingestion_ts") == latest_ts) \
    .select(
        F.col("id").alias("player_id"),
        F.col("web_name").alias("display_name"),
        F.col("first_name"),
        F.col("second_name"),
        F.col("team").alias("team_id"),
        F.col("element_type").alias("position_id"),
        # API stores price in tenths (e.g. 100 = £10.0m)
        (F.col("now_cost") / 10).cast(DoubleType()).alias("current_price"),
        F.col("total_points").cast(IntegerType()),
        F.col("minutes").cast(IntegerType()),
        F.col("goals_scored").cast(IntegerType()),
        F.col("assists").cast(IntegerType()),
        F.col("clean_sheets").cast(IntegerType()),
        F.col("goals_conceded").cast(IntegerType()),
        F.col("bonus").cast(IntegerType()),
        F.col("bps").cast(IntegerType()),
        F.col("influence").cast(DoubleType()),
        F.col("creativity").cast(DoubleType()),
        F.col("threat").cast(DoubleType()),
        F.col("ict_index").cast(DoubleType()),
        F.col("expected_goals").cast(DoubleType()).alias("xG"),
        F.col("expected_assists").cast(DoubleType()).alias("xA"),
        F.col("expected_goal_involvements").cast(DoubleType()).alias("xGI"),
        F.col("selected_by_percent").cast(DoubleType()).alias("ownership_pct"),
        F.col("transfers_in_event").cast(IntegerType()).alias("gw_transfers_in"),
        F.col("transfers_out_event").cast(IntegerType()).alias("gw_transfers_out"),
        F.col("status"),
        F.col("chance_of_playing_next_round"),
        F.col("news"),
        F.col("_ingestion_ts"),
    ) \
    .dropDuplicates(["player_id"])

# Delta MERGE: update existing rows, insert new ones (idempotent upsert)
target_table = "fpl_project.silver.players"

if spark.catalog.tableExists(target_table):
    target = DeltaTable.forName(spark, target_table)
    target.alias("t").merge(
        silver_players.alias("s"),
        "t.player_id = s.player_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
else:
    silver_players.write.format("delta").saveAsTable(target_table)

print("Silver players table updated.")

## Silver Teams and Positions

Simple overwrite — these are small reference tables (~20 teams, 4 positions)
that rarely change, so a full overwrite is simpler than a MERGE.

In [ ]:
for source, target_name, id_col in [
    ("teams", "fpl_project.silver.teams", "id"),
    ("positions", "fpl_project.silver.positions", "id"),
]:
    bronze = spark.table(f"fpl_project.bronze.{source}")
    latest = bronze.agg(F.max("_ingestion_ts")).collect()[0][0]
    silver = bronze.filter(F.col("_ingestion_ts") == latest).dropDuplicates([id_col])

    # Rename 'id' to 'team_id' for teams so joins are unambiguous downstream
    if source == "teams":
        silver = silver.withColumnRenamed("id", "team_id")

    silver.write.mode("overwrite").format("delta").saveAsTable(target_name)

print("Silver teams and positions updated.")

## Silver Fixtures

Rename cryptic API columns (e.g. `team_h` → `home_team_id`, `event` → `gameweek`)
and deduplicate by `fixture_id`. Full overwrite since the fixture list is a
single-season snapshot.

In [ ]:
bronze_fix = spark.table("fpl_project.bronze.fixtures")
latest_ts = bronze_fix.agg(F.max("_ingestion_ts")).collect()[0][0]

silver_fixtures = bronze_fix \
    .filter(F.col("_ingestion_ts") == latest_ts) \
    .select(
        F.col("id").alias("fixture_id"),
        F.col("event").alias("gameweek"),
        F.col("team_h").alias("home_team_id"),
        F.col("team_a").alias("away_team_id"),
        F.col("team_h_score").alias("home_score"),
        F.col("team_a_score").alias("away_score"),
        F.col("team_h_difficulty").alias("home_difficulty"),
        F.col("team_a_difficulty").alias("away_difficulty"),
        F.col("kickoff_time"),
        F.col("finished"),
        F.col("_ingestion_ts"),
    ) \
    .dropDuplicates(["fixture_id"])

silver_fixtures.write.mode("overwrite").format("delta") \
    .saveAsTable("fpl_project.silver.fixtures")

print("Silver fixtures updated.")

## Silver Player History

Gameweek-level stats per player. Uses a composite MERGE key `(player_id, gameweek)`
so re-running after new gameweeks only inserts the new rows while updating any
stat corrections for previously ingested gameweeks.

In [ ]:
silver_history = spark.table("fpl_project.bronze.player_history") \
    .select(
        F.col("player_id"),
        F.col("round").alias("gameweek"),
        F.col("fixture").alias("fixture_id"),
        F.col("opponent_team").alias("opponent_team_id"),
        F.col("was_home").cast("boolean"),
        F.col("total_points").cast(IntegerType()),
        F.col("minutes").cast(IntegerType()),
        F.col("goals_scored").cast(IntegerType()),
        F.col("assists").cast(IntegerType()),
        F.col("clean_sheets").cast(IntegerType()),
        F.col("bonus").cast(IntegerType()),
        F.col("bps").cast(IntegerType()),
        # Price stored in tenths in the API; keep raw here for precision
        F.col("value").cast(IntegerType()).alias("price_at_gw_x10"),
        F.col("selected").cast(IntegerType()).alias("selected_by"),
        F.col("transfers_in").cast(IntegerType()),
        F.col("transfers_out").cast(IntegerType()),
        F.col("expected_goals").cast(DoubleType()).alias("xG"),
        F.col("expected_assists").cast(DoubleType()).alias("xA"),
        F.col("_ingestion_ts"),
    ) \
    .dropDuplicates(["player_id", "gameweek"])

# MERGE on composite key (player_id, gameweek) for idempotent upserts
target_table = "fpl_project.silver.player_history"

if spark.catalog.tableExists(target_table):
    target = DeltaTable.forName(spark, target_table)
    target.alias("t").merge(
        silver_history.alias("s"),
        "t.player_id = s.player_id AND t.gameweek = s.gameweek"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .execute()
else:
    silver_history.write.format("delta").saveAsTable(target_table)

print("Silver player history updated.")